# Industrial-VLM: QLoRA Fine-Tuning Pipeline
**Model**: LLaVA-1.5-7B | **Method**: QLoRA (4-bit NF4) | **Dataset**: MVTec AD (15 categories)

This notebook runs the complete pipeline:
1. Data preprocessing (RGB conversion, 336x336 resize, stratified split)
2. Zero-shot baseline evaluation
3. Weights & Biases authentication
4. QLoRA fine-tuning (r=32, alpha=64, Q-K-V-O projections)
5. Fine-tuned evaluation
6. Auto-push results to GitHub
7. Artifact backup to WandB

**Requirements**: Kaggle GPU T4 x2, MVTec AD dataset attached.

## Cell 1: Setup & Environment Initialization

In [ ]:
import os, shutil

REPO_DIR = "/kaggle/working/vlm-industrial-finetuner"

# Clean previous clone if exists to prevent nested directories
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone https://github.com/dvydinh/vlm-industrial-finetuner.git {REPO_DIR}
!pip install -q -r {REPO_DIR}/requirements.txt
!pip install -q wandb -U

## Cell 2: Data Preprocessing (ETL Pipeline)
Converts MVTec AD from unsupervised format to supervised instruction-tuning JSONL.
- Grayscale -> RGB conversion
- 1024x1024 -> 336x336 resize
- Stratified 80/20 train/test split
- Context-anchored prompts with item category tagging

In [ ]:
!python /kaggle/working/vlm-industrial-finetuner/src/data_builder.py \
    --data_dir /kaggle/input/datasets/ipythonx/mvtec-ad \
    --output_dir /kaggle/working/processed_data

## Cell 3: Zero-Shot Baseline Evaluation
Evaluates the base LLaVA-1.5-7B (without LoRA adapters) to establish a performance baseline.
Metrics are exported to `/kaggle/working/results/eval_baseline.json`.

In [ ]:
!python /kaggle/working/vlm-industrial-finetuner/src/evaluate.py \
    --baseline \
    --test_data /kaggle/working/processed_data

## Cell 4: Weights & Biases Authentication
Provides session credentials for remote logging and artifact storage.
Get your API key from https://wandb.ai/authorize

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))
    print("[SYSTEM] WandB logged in using Kaggle Secrets.")
except Exception as e:
    print("[WARN] Could not find Kaggle Secret WANDB_API_KEY. Please replace manually:")
    wandb.login(key="YOUR_WANDB_API_KEY_HERE")

## Cell 5: QLoRA Fine-Tuning Execution
Executes PEFT configuration targeting Q-K-V-O self-attention modules.
Training logs are periodically exported to `/kaggle/working/results/training_log.json`.

In [ ]:
!python /kaggle/working/vlm-industrial-finetuner/src/train.py \
    --dataset /kaggle/working/processed_data \
    --output_dir /kaggle/working/lora_weights \
    --epochs 2 \
    --lr 2e-5

## Cell 6: Fine-Tuned Model Evaluation
Evaluates the combined Base Model + newly trained LoRA adapters against the test partition.
Metrics are exported to `/kaggle/working/results/eval_finetuned.json`.

In [ ]:
!python /kaggle/working/vlm-industrial-finetuner/src/evaluate.py \
    --model_dir /kaggle/working/lora_weights \
    --test_data /kaggle/working/processed_data

## Cell 7: Repository Metric Synchronization
Uploads the generated evaluation data points (`results/*.json`, `results/*.csv`) directly to the project's main branch.
Note: A personal access token with `repo` scope is required.

In [ ]:
import subprocess, os
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")
except:
    GITHUB_TOKEN = "ghp_YOUR_TOKEN_HERE"

REPO_DIR = "/kaggle/working/vlm-industrial-finetuner"

# Initialize global git configuration
subprocess.run(["git", "config", "--global", "user.email", "dvydinh@users.noreply.github.com"], check=True)
subprocess.run(["git", "config", "--global", "user.name", "dvydinh"], check=True)

# Stage evaluation artifacts into the repository
os.makedirs(f"{REPO_DIR}/results", exist_ok=True)
subprocess.run(f"cp /kaggle/working/results/*.json {REPO_DIR}/results/", shell=True)
subprocess.run(f"cp /kaggle/working/results/*.csv {REPO_DIR}/results/", shell=True)

# Commit and push using cwd parameter (avoids os.chdir side effects)
subprocess.run(["git", "add", "results/"], check=True, cwd=REPO_DIR)
subprocess.run(["git", "commit", "-m", "results: auto-upload evaluation metrics from Kaggle"], check=True, cwd=REPO_DIR)
subprocess.run(["git", "push", f"https://{GITHUB_TOKEN}@github.com/dvydinh/vlm-industrial-finetuner.git", "main"], check=True, cwd=REPO_DIR)
print("[SYSTEM] Evaluation files synchronized to GitHub successfully.")

## Cell 8: WandB Artifact Archival
Generates a consolidated artifact of the Kaggle pipeline output (both weights and numerical logs) for long-term storage.

In [ ]:
import wandb

print("[SYSTEM] Initializing persistent workspace archival to WandB...")
run = wandb.init(project="vlm-industrial-finetuner", name="session_archival")

artifact = wandb.Artifact(name="full-kaggle-workspace", type="backup")
artifact.add_dir("/kaggle/working/lora_weights", name="lora_weights")
artifact.add_dir("/kaggle/working/results", name="results_and_logs")

run.log_artifact(artifact)
run.finish()
print("[SYSTEM] Archival process completed successfully.")